In [2]:
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

def align_and_display(seq1, seq2, name1="Seq1", name2="Seq2"):
    from Bio import pairwise2
    alignments = pairwise2.align.globalxx(seq1, seq2)
    aln1, aln2, score, start, end = alignments[0]

    print(f"\nGlobal Alignment ({name1} vs {name2}):")
    print(f"Score: {score}, Length: {len(aln1)}\n")

    # 标准化 label 宽度
    label_width = max(len(name1), len(name2)) + 1

    # 构建对齐标识行
    match_line = []
    for a, b in zip(aln1, aln2):
        if a == b:
            match_line.append("|")
        elif a == "-" or b == "-":
            match_line.append(" ")
        else:
            match_line.append("*")
    match_line = ''.join(match_line)

    # 分段输出（可调长度）
    step = 60
    for i in range(0, len(aln1), step):
        s1 = aln1[i:i+step]
        ml = match_line[i:i+step]
        s2 = aln2[i:i+step]
        print(f"{name1:<{label_width}}: {s1}")
        print(f"{'':<{label_width}}  {ml}")
        print(f"{name2:<{label_width}}: {s2}\n")


In [3]:
# 示例序列
seq_dqa1_0301 = "DHVASYGVNLYQSYGPSGQYSHEFDGDEEFYVDLERKETVWQLPLFRRFRRFDPQFALTN" \
                "IAVLKHNLNIVIKRSNSTAATN"
seq_dqa1_0201 = "DHVASYGVNLYQSYGPSGQFTHEFDGDEEFYVDLERKETVWKLPLFHRLRFDPQFALTNI" \
                "AVLKHNLNILIKRSNSTAATN"

# 运行比对并显示
align_and_display(seq_dqa1_0301, seq_dqa1_0201, "DQA1*03:01", "DQA1*02:01")


Global Alignment (DQA1*03:01 vs DQA1*02:01):
Score: 75.0, Length: 88

DQA1*03:01 : DHVASYGVNLYQSYGPSGQYS--HEFDGDEEFYVDLERKETVWQ-LPLFRRF-R-RFDPQ
             |||||||||||||||||||    ||||||||||||||||||||  |||   | | |||||
DQA1*02:01 : DHVASYGVNLYQSYGPSGQ--FTHEFDGDEEFYVDLERKETVW-KLPL---FHRLRFDPQ

DQA1*03:01 : FALTNIAVLKHNLNIV-IKRSNSTAATN
             |||||||||||||||  |||||||||||
DQA1*02:01 : FALTNIAVLKHNLNI-LIKRSNSTAATN



In [1]:
import torch
import esm

# 加载模型
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()  # inference 模式

# 示例：输入两个 HLA 氨基酸序列
data = [
    ("DQA1*03:01", "DHVASYGVNLYQSYGPSGQYSHEFDGDEEFYVDLERKETVWQLPLFRRFRRFDPQFALTNIAVLKHNLNIVIKRSNSTAATN"),
    ("DQA1*02:01", "DHVASYGVNLYQSYGPSGQFTHEFDGDEEFYVDLERKETVWKLPLFHRLRFDPQFALTNIAVLKHNLNILIKRSNSTAATN"),
]

batch_labels, batch_strs, batch_tokens = batch_converter(data)

# 前向传播，获取 per-token representation
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[33], return_contacts=False)
    token_representations = results["representations"][33]

# 获取每条序列的平均嵌入（去除首尾的特殊标记）
embeddings = []
for i, (_, seq) in enumerate(data):
    emb = token_representations[i, 1:len(seq)+1].mean(0)
    embeddings.append(emb)

# 计算 cosine similarity
cos_sim = torch.nn.functional.cosine_similarity(embeddings[0], embeddings[1], dim=0)
print(f"Cosine similarity: {cos_sim.item():.4f}")


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to C:\Users\86460/.cache\torch\hub\checkpoints\esm2_t33_650M_UR50D.pt


KeyboardInterrupt: 